# Bonusa tēma 1 - Dabiskās valodas apstrāde ar NLTK un spaCy

Šī piezīmju grāmata ievada praktiskas dabiskās valodas apstrādes (NLP) metodes Python vidē, izmantojot **latviešu valodas piemērus**.

Izmantosim divas bibliotēkas:

- **NLTK** vienkāršai teksta sadalīšanai, filtrēšanai, biežuma analīzei un mācību klasifikācijas piemēriem.
- **spaCy** latviešu teksta tokenizācijai un noteikumos balstītai frāžu un entītiju atpazīšanai ar `spacy.blank("lv")`.

Svarīga piezīme: latviešu valodai NLTK un spaCy nepiedāvā tik plašus gatavus modeļus kā angļu valodai. Tāpēc šajā notebookā apzināti parādīsim gan to, ko var izdarīt uzreiz, gan to, kur nepieciešami latviešu valodai īpaši resursi vai modeļi.


## Nodarbības pārskats

Dabiskās valodas apstrāde ir datorikas nozare, kas nodarbojas ar cilvēku valodas teksta apstrādi, strukturēšanu, analīzi un ģenerēšanu.

Šajā piezīmju grāmatā izmantosim praktisku darba plūsmu:

1. Sākam ar neapstrādātu latviešu tekstu.
2. Sadalām tekstu teikumos un tokenos.
3. Normalizējam tokenus, piemēram, pārveidojam mazos burtos vai filtrējam pieturzīmes.
4. Veicam vienkāršu biežuma analīzi.
5. Izmantojam noteikumos balstītu frāžu un entītiju atrašanu.
6. Salīdzinām NLTK un spaCy iespējas latviešu valodai.

Atšķirībā no angļu versijas, šeit nebūs gatavas NLTK WordNet lematizācijas, NLTK latviešu POS taggera vai oficiāla spaCy trenēta latviešu modeļa. Tas ir būtisks mācību punkts: valodas rīku iespējas ir atkarīgas no konkrētai valodai pieejamiem resursiem.


## Mācību mērķi

Pēc šīs piezīmju grāmatas izpildes jums vajadzētu prast:

- paskaidrot, kas ir tokenizācija, teikumu segmentēšana, pieturvārdi, biežuma analīze un entītiju atpazīšana;
- izmantot NLTK latviešu teksta tokenizācijai, filtrēšanai, vārdu skaitīšanai un vienkāršai klasifikācijai;
- izveidot un pielāgot latviešu pieturvārdu sarakstu;
- izmantot `spacy.blank("lv")` latviešu teksta tokenizācijai;
- pievienot spaCy `sentencizer`, `Matcher` un `EntityRuler` komponentes;
- saprast, kāpēc latviešu valodai bieži vajadzīgi specializēti rīki, piemēram, Stanza modeļi, AiLab resursi vai valodai pielāgoti modeļi.


## Priekšzināšanas

Vēlams pārzināt:

- Python virknes;
- sarakstus, vārdnīcas un kopas;
- ciklus un sarakstu ģeneratorus;
- funkcijas;
- pakotņu instalēšanu ar `pip`;
- Jupyter Notebook vai Google Colab pamatus.


## Google Colab un lokāla uzstādīšana

Ja izmantojat **Google Colab**, palaidiet nākamo uzstādīšanas šūnu. Tā instalē `nltk` un `spacy`, ja tie nav pieejami.

Ja izmantojat lokālu virtuālo vidi, varat instalēt bibliotēkas arī terminālī:

```bash
pip install nltk spacy
```

Šajā latviešu versijā mēs **nelādējam** `en_core_web_sm`, jo tas ir angļu valodas modelis. Tā vietā izmantojam:

```python
import spacy
nlp_lv = spacy.blank("lv")
```

`spacy.blank("lv")` nodrošina latviešu valodas pamatklasi un tokenizāciju, bet nesatur trenētu POS taggeri, lematizētāju, sintaktisko analizatoru vai NER modeli.


In [ ]:
# Colab/lokālās vides uzstādīšanas šūna.
# Palaidiet šo šūnu Google Colab vai svaigā lokālā Jupyter vidē.

import importlib.util
import subprocess
import sys


def ensure_import(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f"Instalēju {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    else:
        print(f"{import_name} jau ir pieejams.")


ensure_import("nltk")
ensure_import("spacy")

print("Uzstādīšanas pārbaude pabeigta.")


## NLTK datu lejupielāde

Šajā notebookā NLTK izmantosim galvenokārt tokenizācijai un pieturvārdu korpusam. Dažas NLTK datu pakotnes ir nepieciešamas atsevišķi.

Latviešu pieturvārdi NLTK vidē var nebūt pieejami visās instalācijās vai versijās, tāpēc tālāk definēsim arī savu nelielu latviešu pieturvārdu sarakstu.


In [ ]:
import nltk

nltk_packages = [
    "punkt",
    "punkt_tab",
    "stopwords",
]

for package in nltk_packages:
    try:
        ok = nltk.download(package, quiet=True)
        status = "ok" if ok else "nav atrasts vai nav pieejams šajā NLTK indeksā"
        print(f"{package}: {status}")
    except Exception as exc:
        print(f"{package}: {exc}")


## Versiju pārbaude

NLP rezultāti var mainīties atkarībā no bibliotēku versijām. Saglabājiet versijas, ja darbs jāpadara reproducējams.


In [ ]:
import nltk
import spacy

print("NLTK versija:", nltk.__version__)
print("spaCy versija:", spacy.__version__)

nlp_lv = spacy.blank("lv")
print("spaCy valoda:", nlp_lv.lang)
print("Sākotnējās pipeline komponentes:", nlp_lv.pipe_names)


## Latviešu piemēru teksti

Izmantosim nelielus latviešu valodas tekstus, lai notebooku var palaist bez ārējiem failiem.


In [ ]:
main_text = (
    "2026. gadā Anna pievienojās uzņēmumam Baltijas Robotika Rīgā. "
    "Viņa veido Python rīkus latviešu valodas datiem un māca programmēšanu Latvijas Universitātē."
)

review_text = (
    "Šī Python piezīmju grāmata ir praktiska un saprotama, "
    "bet uzstādīšanas soļi ir nedaudz gari. Kopumā es ieteiktu šo kursu iesācējiem."
)

news_text = (
    "Latvijas Universitāte pirmdien atvēra jaunu pētniecības centru Rīgā. "
    "Tēzaurs komanda publicēja jaunu valodas resursu aprakstu."
)

email_text = (
    "Labdien! Lūdzu, nosūtiet NLP pārskatu Annai līdz piektdienai. "
    "Klients no Rīgas gaida īsu kopsavilkumu par latviešu valodas datiem."
)

print(main_text)


## Latviešu valodas īpatnības NLP uzdevumos

Latviešu valoda ir morfoloģiski bagāta. Tas nozīmē, ka viena vārdnīcas forma tekstā var parādīties daudzās locījumu un gramatiskās formās.

| Pamatforma | Teksta formas |
| --- | --- |
| `Rīga` | `Rīga`, `Rīgā`, `Rīgas`, `Rīgai` |
| `valoda` | `valoda`, `valodas`, `valodai`, `valodu` |
| `students` | `students`, `studenti`, `studentam`, `studentu` |
| `mācīt` | `māca`, `mācīja`, `mācīs`, `mācīt` |

Tāpēc latviešu tekstam ar vienkāršu `lower()` un galotņu nogriešanu bieži nepietiek. Reālai lematizācijai un morfoloģiskai analīzei vajadzīgi valodai pielāgoti rīki.


# 1. daļa: NLTK latviešu tekstam

NLTK ir noderīgs arī latviešu piemēriem, ja izmantojam valodneitrālas metodes:

- tokenizāciju;
- pieturzīmju filtrēšanu;
- mazo burtu normalizāciju;
- vārdu biežuma analīzi;
- pielāgotu pieturvārdu sarakstu;
- vienkāršu klasifikāciju.

Tomēr NLTK standarta angļu resursi nav jāuzskata par latviešu valodas resursiem.


## NLTK: tokenizācija

Sāksim ar NLTK `word_tokenize`. Tas nav īpaši trenēts latviešu valodai, bet vienkāršiem piemēriem bieži ir pietiekams sākumpunkts.


In [ ]:
from nltk.tokenize import word_tokenize

nltk_tokens = word_tokenize(main_text)
print(nltk_tokens)


### Salīdzinājums ar `split()`

`str.split()` sadala tekstu tikai pēc atstarpēm. Tokenizators atdala arī daļu pieturzīmju.


In [ ]:
print("split():")
print(main_text.split()[:20])

print("\nword_tokenize():")
print(word_tokenize(main_text)[:25])


## NLTK: regulārās izteiksmes tokenizators

Latviešu tekstam jāņem vērā diakritiskie burti: `ā`, `č`, `ē`, `ģ`, `ī`, `ķ`, `ļ`, `ņ`, `š`, `ū`, `ž` un to lielie burti.

Vienkāršs angļu paraugs `[A-Za-z]+` latviešu tekstu sabojātu, jo neiekļautu latviešu burtus.


In [ ]:
from nltk.tokenize import RegexpTokenizer

latvian_word_pattern = r"[A-Za-zĀČĒĢĪĶĻŅŠŪŽāčēģīķļņšūž]+"
latvian_tokenizer = RegexpTokenizer(latvian_word_pattern)

lv_word_tokens = latvian_tokenizer.tokenize(main_text)
print(lv_word_tokens)


## NLTK: mazie burti un pielāgoti pieturvārdi

Pieturvārdi ir bieži vārdi, kurus dažos uzdevumos filtrē, piemēram, biežuma analīzē.

Latviešu valodai šeit definēsim nelielu mācību pieturvārdu sarakstu. Reālos projektos tas būtu jāpārbauda un jāpielāgo konkrētajam uzdevumam.


In [ ]:
custom_lv_stopwords = {
    "aiz", "ap", "ar", "arī", "bet", "bija", "būt", "es", "gan", "ir",
    "ja", "jau", "jo", "ka", "kā", "kas", "ko", "kur", "lai", "līdz",
    "man", "mēs", "nav", "ne", "no", "par", "pie", "pēc", "šo", "šī",
    "tas", "tā", "tāpat", "te", "tie", "tiek", "tikai", "to", "un", "uz",
    "vai", "var", "viņa", "viņš", "vis", "vēl",
}

review_tokens = latvian_tokenizer.tokenize(review_text)
filtered_review_tokens = [
    token.lower()
    for token in review_tokens
    if token.lower() not in custom_lv_stopwords
]

print("Sākotnējie vārdu tokeni:")
print(review_tokens)

print("\nPēc pieturvārdu filtrēšanas:")
print(filtered_review_tokens)


### Vai NLTK instalācijā ir latviešu pieturvārdi?

Dažās NLTK datu versijās var nebūt `latvian` pieturvārdu saraksta. Šūna zemāk droši pārbauda pieejamību.


In [ ]:
from nltk.corpus import stopwords

try:
    available_stopword_languages = stopwords.fileids()
    print("Pieejamās stopwords valodas:")
    print(available_stopword_languages)
    if "latvian" in available_stopword_languages:
        nltk_lv_stopwords = set(stopwords.words("latvian"))
        print("\nNLTK latviešu pieturvārdu piemēri:", sorted(nltk_lv_stopwords)[:30])
    else:
        print("\nŠajā NLTK instalācijā 'latvian' pieturvārdu saraksts nav pieejams.")
except LookupError as exc:
    print("Stopwords resurss nav lejupielādēts:", exc)


## NLTK: biežuma analīze

Pēc tokenizācijas un filtrēšanas varam saskaitīt biežākos vārdus.


In [ ]:
from nltk.probability import FreqDist

all_demo_text = " ".join([main_text, review_text, news_text, email_text])
all_tokens = latvian_tokenizer.tokenize(all_demo_text)
content_tokens = [
    token.lower()
    for token in all_tokens
    if token.lower() not in custom_lv_stopwords
]

fdist = FreqDist(content_tokens)
print(fdist.most_common(15))


## NLTK: kāpēc angļu stemmeri neder latviešu valodai

NLTK piedāvā angļu un citu valodu stemmerus, taču tos nedrīkst automātiski lietot latviešu tekstam. Latviešu galotņu un morfoloģijas sistēma ir cita.

Zemāk redzamais piemērs ir **tikai mācību demonstrācija**, nevis īsts latviešu stemmeris.


In [ ]:
def toy_latvian_suffix_strip(word):
    endings = ["iem", "ām", "os", "us", "as", "ai", "am", "ā", "a", "u", "i", "s"]
    lowered = word.lower()
    for ending in endings:
        if lowered.endswith(ending) and len(lowered) > len(ending) + 3:
            return lowered[: -len(ending)]
    return lowered


examples = ["Rīga", "Rīgā", "Rīgas", "studentiem", "valodas", "datiem", "kursu", "programmēšana"]

for word in examples:
    print(f"{word:15} -> {toy_latvian_suffix_strip(word)}")


Šī pieeja var šķist noderīga, bet tā viegli kļūdās. Piemēram, īsta lematizācija prasītu atšķirt vārdšķiru, locījumu, skaitli un citas pazīmes. Latviešu lematizācijai praktiskāk izmantot specializētus rīkus, piemēram, Stanza latviešu modeļus vai AiLab rīkus.


## NLTK: vienkāršs atslēgvārdu izvilkšanas piemērs

Ar NLTK un parastiem Python rīkiem varam izveidot vienkāršu atslēgvārdu sarakstu.


In [ ]:
def simple_keywords(text, stopwords_set, top_n=10):
    tokens = latvian_tokenizer.tokenize(text)
    content = [
        token.lower()
        for token in tokens
        if len(token) > 2 and token.lower() not in stopwords_set
    ]
    return FreqDist(content).most_common(top_n)


print(simple_keywords(all_demo_text, custom_lv_stopwords, top_n=12))


## NLTK: vienkārša teksta klasifikācija latviski

Šis ir mazs mācību piemērs. Reālam klasifikatoram vajag vairāk datu, atsevišķu testēšanas kopu un kvalitātes metriku.


In [ ]:
from nltk.classify import NaiveBayesClassifier

training_data = [
    ("Šī nodarbība ir skaidra un noderīga", "pozitīvs"),
    ("Piemēri ir praktiski un saprotami", "pozitīvs"),
    ("Man patīk šis Python kurss", "pozitīvs"),
    ("Uzdevumi palīdz labāk saprast tēmu", "pozitīvs"),
    ("Šī instalācija ir mulsinoša", "negatīvs"),
    ("Piemērs ir pārāk sarežģīts", "negatīvs"),
    ("Uzdevums ir garš un neskaidrs", "negatīvs"),
    ("Man nepatīk šis skaidrojums", "negatīvs"),
]


def document_features_lv(sentence):
    words = set(token.lower() for token in latvian_tokenizer.tokenize(sentence))
    return {
        "ir_skaidrs": "skaidra" in words or "skaidrs" in words,
        "ir_noderīgs": "noderīga" in words or "noderīgs" in words,
        "ir_praktisks": "praktiski" in words or "praktisks" in words,
        "ir_sarežģīts": "sarežģīts" in words or "sarežģīti" in words,
        "ir_neskaidrs": "neskaidrs" in words or "neskaidra" in words,
        "ir_pārāk": "pārāk" in words,
        "ir_nepatīk": "nepatīk" in words,
        "ir_patīk": "patīk" in words,
    }


featuresets = [(document_features_lv(sentence), label) for sentence, label in training_data]
classifier_lv = NaiveBayesClassifier.train(featuresets)

test_sentences = [
    "Šis Python piemērs ir skaidrs",
    "Uzdevums ir pārāk sarežģīts",
    "Nodarbība ir praktiska un noderīga",
]

for sentence in test_sentences:
    print(sentence, "->", classifier_lv.classify(document_features_lv(sentence)))

print("\nInformatīvākās pazīmes:")
classifier_lv.show_most_informative_features()


# 2. daļa: spaCy latviešu tekstam

spaCy latviešu valodai var izmantot ar `spacy.blank("lv")`.

Tas dod:

- latviešu valodas pamatklasi;
- tokenizāciju;
- iespēju pievienot komponentes, piemēram, `sentencizer`, `Matcher`, `EntityRuler`.

Tas nedod gatavu trenētu:

- POS taggeri;
- lematizētāju;
- sintaktisko analizatoru;
- NER modeli.


## spaCy: tukša latviešu pipeline


In [ ]:
import spacy

nlp_lv = spacy.blank("lv")
print("Valodas kods:", nlp_lv.lang)
print("Pipeline komponentes:", nlp_lv.pipe_names)


## spaCy: tokenizācija

Pat bez trenēta modeļa spaCy var sadalīt latviešu tekstu tokenos.


In [ ]:
doc_lv = nlp_lv(main_text)

for token in doc_lv:
    print(f"{token.text:18} idx={token.idx:<3} alpha={token.is_alpha} punct={token.is_punct}")


## spaCy: teikumu segmentēšana ar `sentencizer`

Tukšai pipeline nav sintaktiskā analizatora, tāpēc pievienosim noteikumos balstītu teikumu segmentētāju.


In [ ]:
# Izveidojam svaigu pipeline, lai šo šūnu var droši palaist atkārtoti.
nlp_lv_sent = spacy.blank("lv")

if "sentencizer" not in nlp_lv_sent.pipe_names:
    nlp_lv_sent.add_pipe("sentencizer")

sent_doc = nlp_lv_sent(main_text)

for sent in sent_doc.sents:
    print(sent.text)


## spaCy: ko tukša pipeline neparedz

Tukša latviešu pipeline neveic POS tagošanu, lematizāciju, sintaktisko analīzi vai NER. Atribūti tehniski pastāv, bet nav aizpildīti ar trenēta modeļa prognozēm.


In [ ]:
for token in doc_lv[:12]:
    print(
        f"{token.text:18} pos={token.pos_!r:6} lemma={token.lemma_!r:12} dep={token.dep_!r}"
    )


## spaCy: `Matcher` latviešu frāzēm

`Matcher` ļauj atrast tokenu secības pēc noteikumiem. Tas ir noderīgi, ja meklējam konkrētas frāzes vai terminus.


In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp_lv.vocab)

matcher.add("PYTHON_TERMINS", [
    [{"LOWER": "python"}, {"LOWER": {"IN": ["kurss", "rīkus", "piezīmju"]}}],
    [{"LOWER": "latviešu"}, {"LOWER": "valodas"}, {"LOWER": "datiem"}],
    [{"LOWER": "nlp"}, {"LOWER": "pārskatu"}],
])

matcher_doc = nlp_lv_sent("Šis Python kurss izmanto Python rīkus latviešu valodas datiem. Lūdzu, nosūtiet NLP pārskatu.")

for match_id, start, end in matcher(matcher_doc):
    label = nlp_lv.vocab.strings[match_id]
    span = matcher_doc[start:end]
    print(label, "->", span.text)


## spaCy: `EntityRuler` noteikumos balstītām entītijām

Tā kā mums nav gatava latviešu NER modeļa, varam parādīt noteikumos balstītu entītiju atpazīšanu ar `EntityRuler`.


In [ ]:
# Izveidojam svaigu pipeline ar sentencizer un entity_ruler.
nlp_lv_rules = spacy.blank("lv")
nlp_lv_rules.add_pipe("sentencizer")
ruler = nlp_lv_rules.add_pipe("entity_ruler")

patterns = [
    {"label": "PERSON", "pattern": "Anna"},
    {"label": "PERSON", "pattern": "Annai"},
    {"label": "GPE", "pattern": "Rīga"},
    {"label": "GPE", "pattern": "Rīgā"},
    {"label": "ORG", "pattern": "Latvijas Universitāte"},
    {"label": "ORG", "pattern": "Latvijas Universitātē"},
    {"label": "ORG", "pattern": "Baltijas Robotika"},
    {"label": "RESOURCE", "pattern": "Tēzaurs"},
    {"label": "TOPIC", "pattern": "latviešu valodas dati"},
    {"label": "TOPIC", "pattern": "latviešu valodas datiem"},
]

ruler.add_patterns(patterns)

rule_doc = nlp_lv_rules(" ".join([main_text, news_text, email_text]))

for ent in rule_doc.ents:
    print(f"{ent.text:30} -> {ent.label_}")


## spaCy: vienkārša entītiju vizualizācija

`displacy` var attēlot entītijas arī tad, ja tās pievienotas ar `EntityRuler`.


In [ ]:
from spacy import displacy

short_rule_doc = nlp_lv_rules("Anna strādā Latvijas Universitātē Rīgā un izmanto Tēzaurs resursus.")
displacy.render(short_rule_doc, style="ent", jupyter=True)


## spaCy: lielo sākumburtu kandidāti

Dažreiz pirms pilna NER modeļa izmantošanas var noderēt kandidātu atlase. Zemāk ir vienkāršs, nepilnīgs piemērs, kas meklē secīgus vārdus ar lielo sākumburtu.

Tas nav īsts NER risinājums, jo teikuma sākuma vārdi un virsraksti var radīt daudz kļūdainu kandidātu.


In [ ]:
def capitalized_spans(doc):
    spans = []
    current = []
    for token in doc:
        if token.is_alpha and token.text[:1].isupper():
            current.append(token)
        else:
            if current:
                spans.append(doc[current[0].i : current[-1].i + 1])
                current = []
    if current:
        spans.append(doc[current[0].i : current[-1].i + 1])
    return spans


candidate_doc = nlp_lv(news_text)
for span in capitalized_spans(candidate_doc):
    print(span.text)


## spaCy: paketveida apstrāde ar `nlp.pipe()`

Ja jāapstrādā vairāki teksti, izmantojiet `nlp.pipe()`.


In [ ]:
texts = [main_text, review_text, news_text, email_text]

for processed_doc in nlp_lv_rules.pipe(texts):
    entities = [(ent.text, ent.label_) for ent in processed_doc.ents]
    print(processed_doc.text[:70] + "...")
    print("  Entītijas:", entities)


# 3. daļa: NLTK un spaCy salīdzinājums latviešu valodai

| Uzdevums | NLTK | spaCy |
| --- | --- | --- |
| Vienkārša tokenizācija | Var izmantot `word_tokenize` vai `RegexpTokenizer` | Var izmantot `spacy.blank("lv")` |
| Pieturvārdu filtrēšana | Var izmantot pielāgotu sarakstu | Var izmantot pielāgotu sarakstu vai tokenu atribūtus, ja tie pieejami |
| Biežuma analīze | Ļoti ērti ar `FreqDist` | Var darīt ar Python kolekcijām |
| Stemming | Nav gatava uzticama latviešu risinājuma NLTK standartā | Nav tukšā pipeline |
| Lemmatizācija | Nav gatava NLTK WordNet tipa risinājuma latviski | Nav oficiāla trenēta latviešu pipeline |
| POS tagošana | Nav gatava latviešu taggera NLTK standartā | Nav tukšā pipeline |
| NER | Nav gatava latviešu NER NLTK standartā | Var lietot `EntityRuler`; trenēts modelis būtu jāiegūst citur vai jātrenē |
| Noteikumos balstīta frāžu atrašana | Var darīt ar Python/regex | Ļoti ērti ar `Matcher` un `EntityRuler` |


## Tokenizācijas salīdzinājums


In [ ]:
comparison_text = "Anna Rīgā māca Python kursu un analizē latviešu valodas datus."

print("NLTK RegexpTokenizer:")
print(latvian_tokenizer.tokenize(comparison_text))

print("\nspaCy blank('lv'):")
print([token.text for token in nlp_lv(comparison_text)])


## Entītiju salīdzinājums ar noteikumiem

NLTK šajā notebookā entītijas varam meklēt ar vienkāršiem Python noteikumiem, bet spaCy `EntityRuler` dod ērtāku struktūru un `Doc.ents` rezultātu.


In [ ]:
def simple_rule_entities(text):
    known = {
        "Latvijas Universitāte": "ORG",
        "Latvijas Universitātē": "ORG",
        "Baltijas Robotika": "ORG",
        "Rīgā": "GPE",
        "Rīga": "GPE",
        "Anna": "PERSON",
        "Annai": "PERSON",
        "Tēzaurs": "RESOURCE",
    }
    found = []
    for phrase, label in known.items():
        if phrase in text:
            found.append((phrase, label))
    return found


sample_entity_text = "Anna no Latvijas Universitātes strādā Rīgā un izmanto Tēzaurs API."

print("Vienkārši Python noteikumi:")
print(simple_rule_entities(sample_entity_text))

print("\nspaCy EntityRuler:")
for ent in nlp_lv_rules(sample_entity_text).ents:
    print(ent.text, "->", ent.label_)


# 4. daļa: praktiskie uzdevumi

Izpildiet šos uzdevumus pēc demonstrācijas šūnām.


## 1. uzdevums: tokenizācija un biežums

Izmantojiet `email_text`:

1. Sadaliet tekstu vārdos ar `latvian_tokenizer`.
2. Pārveidojiet vārdus mazos burtos.
3. Izfiltrējiet `custom_lv_stopwords`.
4. Izdrukājiet 10 biežākos vārdus.


In [ ]:
# 1. uzdevuma sākuma kods
exercise_tokens = latvian_tokenizer.tokenize(email_text)

# TODO: filtrējiet, pārveidojiet mazos burtos un saskaitiet ar FreqDist
exercise_tokens[:10]


## 2. uzdevums: pieturvārdu saraksta pielāgošana

Papildiniet `custom_lv_stopwords` ar vārdiem, kas jūsu tekstā nav informatīvi. Pēc tam atkārtojiet biežuma analīzi.


In [ ]:
# 2. uzdevuma sākuma kods
my_stopwords = set(custom_lv_stopwords)

# TODO: pievienojiet savus pieturvārdus ar my_stopwords.update([...])
my_stopwords


## 3. uzdevums: lielo sākumburtu kandidāti

Izmantojiet `capitalized_spans`, lai atrastu kandidātus tekstā `email_text`. Kuri kandidāti ir īstas entītijas, un kuri ir kļūdaini?


In [ ]:
# 3. uzdevuma sākuma kods
email_candidates = capitalized_spans(nlp_lv(email_text))

# TODO: izdrukājiet un novērtējiet kandidātus
email_candidates


## 4. uzdevums: EntityRuler papildināšana

Papildiniet `EntityRuler` ar saviem noteikumiem, piemēram:

- `Latvijas Nacionālā bibliotēka` kā `ORG`;
- `Liepājā` kā `GPE`;
- `Python kurss` kā `COURSE`.


In [ ]:
# 4. uzdevuma sākuma kods
exercise_nlp = spacy.blank("lv")
exercise_nlp.add_pipe("sentencizer")
exercise_ruler = exercise_nlp.add_pipe("entity_ruler")

exercise_ruler.add_patterns([
    # TODO: pievienojiet savus pattern objektus
])

exercise_doc = exercise_nlp("Python kurss notiek Rīgā un Liepājā.")
[(ent.text, ent.label_) for ent in exercise_doc.ents]


## 5. uzdevums: Matcher frāžu meklēšanai

Izveidojiet `Matcher` noteikumus, kas atrod:

- `latviešu valodas dati`;
- `NLP pārskats`;
- `Python kurss`.


In [ ]:
# 5. uzdevuma sākuma kods
exercise_matcher = Matcher(nlp_lv.vocab)

# TODO: pievienojiet matcher.add(...) noteikumus


## 6. uzdevums: mazs klasifikators latviski

Izveidojiet vismaz 12 teikumus: 6 pozitīvus un 6 negatīvus. Trenējiet NLTK Naive Bayes klasifikatoru un pārbaudiet to uz 3 jauniem teikumiem.

Padomājiet:

- kuri vārdi ir labas pazīmes;
- vai locījumi rada problēmas;
- vai `nepatīk` un `patīk` tiek pareizi atšķirti.


In [ ]:
# 6. uzdevuma sākuma kods
student_training_data = [
    # ("teikums", "etiķete"),
]

# TODO: izveidojiet pazīmes, trenējiet klasifikatoru un testējiet to


# Papildu piezīme: specializēti latviešu NLP rīki

Šis notebooks ir apzināti būvēts ar NLTK un spaCy, jo mērķis ir parādīt šīs divas Python bibliotēkas. Tomēr latviešu valodai sarežģītākiem uzdevumiem bieži vajadzīgi specializēti resursi.

Noderīgi virzieni tālākai izpētei:

- **Stanza**: Stanford NLP rīks ar latviešu modeļiem tokenizācijai, POS/morfoloģijai, lematizācijai un dependency parsing.
- **Universal Dependencies Latvian LVTB**: latviešu sintaktiski un morfoloģiski anotēts treebank.
- **AiLab resursi**: LU MII Mākslīgā intelekta laboratorijas latviešu valodas resursi, LV-PIPE, Tēzaurs API un citi rīki.

Ja šī kursa nākamais paplašinājums būtu tieši latviešu NLP, būtu vērts izveidot atsevišķu notebooku ar Stanza vai AiLab rīkiem.


# Kopsavilkums

Latviešu valodai NLTK un spaCy jāizmanto uzmanīgi.

NLTK labi der:

- vienkāršai tokenizācijai;
- regulāro izteiksmju tokenizācijai;
- pieturvārdu filtrēšanai;
- vārdu biežuma analīzei;
- maziem klasifikācijas piemēriem.

spaCy ar `spacy.blank("lv")` labi der:

- tokenizācijai;
- teikumu segmentēšanai ar `sentencizer`;
- frāžu meklēšanai ar `Matcher`;
- noteikumos balstītai entītiju atpazīšanai ar `EntityRuler`;
- vairāku tekstu apstrādei ar `nlp.pipe()`.

Latviešu POS tagošana, lematizācija, sintaktiskā analīze un trenēta NER parasti prasa specializētus modeļus vai rīkus.


# Atsauces

Bibliotēku dokumentācija:

- NLTK dokumentācija: <https://www.nltk.org/>
- NLTK datu instalēšana: <https://www.nltk.org/data.html>
- NLTK korpusu dokumentācija: <https://www.nltk.org/howto/corpus.html>
- spaCy modeļi un valodas: <https://spacy.io/usage/models>
- spaCy trenētie modeļi: <https://spacy.io/models>
- spaCy processing pipelines: <https://spacy.io/usage/processing-pipelines>
- spaCy rule-based matching: <https://spacy.io/usage/rule-based-matching>
- spaCy visualizers: <https://spacy.io/usage/visualizers>

Latviešu valodas NLP resursi:

- Stanza pieejamie modeļi: <https://stanfordnlp.github.io/stanza/available_models.html>
- Stanza darba sākšana: <https://stanfordnlp.github.io/stanza/getting_started.html>
- UD Latvian LVTB: <https://universaldependencies.org/treebanks/lv_lvtb/index.html>
- Universal Dependencies latviešu valodas lapa: <https://universaldependencies.org/lv/index.html>
- AiLab resursi: <https://ailab.lv/en/resources/>
- Tēzaurs API: <https://api.tezaurs.lv/>
- Latvian Treebank: <https://sintakse.korpuss.lv/>
